# Data Wrangling / Data Munging / Data handling

The steps related to this script:
In this part we will  handle the following steps :
1. Loading libraries - **Please note that, installation of libraries are mandatory prior to their import. Installation of the libraries has not been shown.(Please refer to the documentations for installing libraries in one's system.)**
2. Create connnection with sQL and load data from SQL
3. Download the table in form of csv file for analysis
4. After Analysis as an immediate step, missing data and regeneration of data is done.

## 1. Loading libraries
The following libraries are loaded:

* [`pandas`](https://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for managing the data, for time-series analysis and holiday calendar support
* [`NumPy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations
* [`SQLAlchemy`](https://www.sqlalchemy.org/) for SQL toolkit and Object-Relational Mapping (ORM)
* [`urllib`](https://docs.python.org/3/library/urllib.html) for handling URLs and making HTTP requests
* [`warnings`](https://docs.python.org/3/library/warnings.html) for handling and suppressing warning messages
* [`pyodbc`](https://pypi.org/project/pyodbc/) for connecting to databases using Open Database Connectivity (ODBC)

In [257]:
import pyodbc
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import urllib
print("libraries loaded sucessfully")

libraries loaded sucessfully


# 2. Creating connection with SQL 

In [258]:
import pyodbc
import pandas as pd
import warnings
from sqlalchemy import create_engine

# 1. Suppress specific SQLAlchemy warnings for localdb versions
warnings.filterwarnings('ignore', message='.*Unrecognized server version info.*')

# 2. Connection Parameters
SERVER = '(localdb)\\FMCGretaildb'
DATABASE = 'FMCGdb'
DRIVER = '{ODBC Driver 17 for SQL Server}'

# 3. Build Connection String & Engine
# We use the creator pattern to handle the Windows Authentication (Trusted_Connection)
conn_str = f'DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes'
engine = create_engine("mssql+pyodbc://", creator=lambda: pyodbc.connect(conn_str))

# 4. Execute Query and Load Data
try:
    # Using 'with' ensures the connection is closed automatically
    with engine.connect() as connection:
        print("Connection established successfully!")
        
        query = "SELECT * FROM load.consolidated_sales_info"
        df = pd.read_sql(query, connection)
except Exception as e:
    print(f"Error occurred: {e}")
    


Connection established successfully!


### 2.1. Checking the consolidated data load in the data frame :

In [259]:
df.head()

,date,sku,brand,segment,category,channel,region,pack_type,price_unit,promotion_flag,...,inflation_index,school_in_session,category_trend,event_score,stockout_flag,inventory_turnover_ratio,days_to_next_delivery,promo_intensity,is_npi_flag,predicted_units_sold
0,2023-12-29,YO-012,YoBrand1,Yogurt-Seg2,Yogurt,E-commerce,PL-North,Single,3.65,0,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
1,2023-12-29,YO-012,YoBrand1,Yogurt-Seg2,Yogurt,E-commerce,PL-South,Carton,6.75,0,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
2,2023-12-29,SN-013,SnBrand3,SnackBar-Seg2,SnackBar,Retail,PL-Central,Carton,3.49,0,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
3,2023-12-29,SN-013,SnBrand3,SnackBar-Seg2,SnackBar,Retail,PL-North,Carton,7.46,0,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None
4,2023-12-29,SN-013,SnBrand3,SnackBar-Seg2,SnackBar,Retail,PL-South,Single,2.25,0,...,NaN,NaN,NaN,NaN,None,None,None,None,None,None


In [260]:
df.columns

Index(['date', 'sku', 'brand', 'segment', 'category', 'channel', 'region',
       'pack_type', 'price_unit', 'promotion_flag', 'delivery_days',
       'stock_available', 'delivered_qty', 'units_sold', 'is_holiday_peak',
       'week_number', 'month_m', 'year_y', 'is_holiday_week', 'is_summer',
       'is_winter', 'sku_age', 'lifecycle_stage', 'lag_1', 'lag_2',
       'rolling_mean_4', 'rolling_std_4', 'momentum', 'target_next_week',
       'price_avg', 'promo_rate', 'stock_avg', 'deliveries', 'avg_temp',
       'inflation_index', 'school_in_session', 'category_trend', 'event_score',
       'stockout_flag', 'inventory_turnover_ratio', 'days_to_next_delivery',
       'promo_intensity', 'is_npi_flag', 'predicted_units_sold'],
      dtype='str')

In [261]:
df.shape

(191181, 44)

In [262]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 191181 entries, 0 to 191180
Data columns (total 44 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   date                      191181 non-null  object 
 1   sku                       191181 non-null  str    
 2   brand                     191181 non-null  str    
 3   segment                   190418 non-null  str    
 4   category                  191181 non-null  str    
 5   channel                   191181 non-null  str    
 6   region                    191181 non-null  str    
 7   pack_type                 191181 non-null  str    
 8   price_unit                191181 non-null  float64
 9   promotion_flag            191181 non-null  int64  
 10  delivery_days             191181 non-null  float64
 11  stock_available           191181 non-null  float64
 12  delivered_qty             191181 non-null  float64
 13  units_sold                191181 non-null  float64
 14 

In [263]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {num_duplicates}")

Total duplicate rows: 0


In [264]:
df.isnull().sum()

date                             0
sku                              0
brand                            0
segment                        763
category                         0
channel                          0
region                           0
pack_type                        0
price_unit                       0
promotion_flag                   0
delivery_days                    0
stock_available                  0
delivered_qty                    0
units_sold                       0
is_holiday_peak             190418
week_number                      0
month_m                          0
year_y                           0
is_holiday_week             190418
is_summer                        0
is_winter                        0
sku_age                     182536
lifecycle_stage             182536
lag_1                       190418
lag_2                       190418
rolling_mean_4              190418
rolling_std_4               190418
momentum                    190418
target_next_week    

# 3. Dowload the data set in csv format before further computations

In [265]:
df.to_csv(r'D:\Documents\Maloshree\DATA ANALYTICS CERTIFICATION-INCO\Capstone Project\Data\consolidated_data_load.csv', index=False)

# 4. Handing missing data and regeneration

### 4.1. Null handling for 'pack_type'

In [266]:
# 1. Clean the pack_type column to remove hidden characters like \r
df['pack_type'] = df['pack_type'].str.strip()

# 2. Update the function to look for empty strings (or None)
def fill_pack_type(row):
    # Strip whitespace from category and brand for safety
    cat = str(row['category']).strip()
    brand = str(row['brand']).strip()
    
    # Check if pack_type is empty (after our strip above, \r becomes '')
    # We also check for 'N/A' just in case some rows have it
    is_empty = row['pack_type'] == '' or pd.isna(row['pack_type']) or row['pack_type'] == 'N/A'
    
    if cat == 'Milk' and brand == 'MiBrand1' and is_empty:
        return 'Single'
    
    return row['pack_type']

# Apply the function
df['pack_type'] = df.apply(fill_pack_type, axis=1)

In [267]:
df['pack_type'].isnull().sum()

np.int64(0)

### 4.2. Null handling for 'segment'

In [268]:
def fill_segment_type(row1):
    if row1['category'] == 'Milk' and row1['brand'] == 'MiBrand1' and pd.isna(row1['segment']):
        return 'Milk-Seg3'
    else:
        return row1['segment']

df['segment'] = df.apply(fill_segment_type, axis=1)

In [269]:
df['pack_type'].isnull().sum()

np.int64(0)

### 4.3. Null handling for 'is_holiday'

In [270]:
from pandas.tseries.holiday import USFederalHolidayCalendar
df['date'] = pd.to_datetime(df['date'])
cal = USFederalHolidayCalendar()
us_holidays = cal.holidays(start='2022-01-01', end='2025-12-31')
df['is_holiday'] = df['date'].isin(us_holidays).astype(int)
print(df[['date', 'is_holiday']].head())

        date  is_holiday
0 2023-12-29           0
1 2023-12-29           0
2 2023-12-29           0
3 2023-12-29           0
4 2023-12-29           0


### 4.4. Null handling for 'is_holiday_week'

In [271]:
df['is_holiday_week'] = df.groupby(['year_y', 'week_number'])['is_holiday'].transform('max')
print(df[['date', 'is_holiday','is_holiday_week']].head())

        date  is_holiday  is_holiday_week
0 2023-12-29           0                1
1 2023-12-29           0                1
2 2023-12-29           0                1
3 2023-12-29           0                1
4 2023-12-29           0                1


### 4.5. Null handling for 'is_holiday_peak'

In [272]:
df['is_holiday_peak'] = (
    df['is_holiday']
    .shift(-7)                 # Start looking 7 days ahead
    .rolling(window=9, min_periods=1) # Look at the 9-day block (7, 8... 15)
    .max()                     # If any of those days is a 1, result is 1
    .fillna(0)                 
    .astype(int)               # Ensure it's int
)

In [273]:
print(df[df['is_holiday_peak'] == 1])

             date     sku     brand        segment  category     channel  \
591    2023-12-31  SN-030  SnBrand2  SnackBar-Seg1  SnackBar      Retail   
592    2023-12-31  SN-030  SnBrand2  SnackBar-Seg1  SnackBar    Discount   
593    2023-12-31  SN-030  SnBrand2  SnackBar-Seg1  SnackBar    Discount   
594    2023-12-31  SN-030  SnBrand2  SnackBar-Seg1  SnackBar    Discount   
595    2023-12-31  SN-030  SnBrand2  SnackBar-Seg1  SnackBar  E-commerce   
...           ...     ...       ...            ...       ...         ...   
190396 2023-12-25  SN-030  SnBrand2  SnackBar-Seg1  SnackBar    Discount   
190397 2023-12-25  SN-030  SnBrand2  SnackBar-Seg1  SnackBar    Discount   
190398 2023-12-25  SN-030  SnBrand2  SnackBar-Seg1  SnackBar  E-commerce   
190399 2023-12-25  SN-030  SnBrand2  SnackBar-Seg1  SnackBar  E-commerce   
190400 2023-12-26  YO-001  YoBrand3    Yogurt-Seg1    Yogurt      Retail   

            region  pack_type  price_unit  promotion_flag  ...  \
591       PL-South  M

### 4.6. Null handling for 'lifecycle_stage'

In [274]:
def fill_lifecycle_stage(row):
    if pd.isna(row['lifecycle_stage']):
        return 'Introduction'
    else:
        # Return the original value if it's already there (e.g., 'Growth')
        return row['lifecycle_stage']
df['lifecycle_stage'] = df.apply(fill_lifecycle_stage, axis=1)

In [275]:
print(df[df['lifecycle_stage']=='Introduction'])

             date     sku     brand        segment  category     channel  \
0      2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt  E-commerce   
1      2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt  E-commerce   
2      2023-12-29  SN-013  SnBrand3  SnackBar-Seg2  SnackBar      Retail   
3      2023-12-29  SN-013  SnBrand3  SnackBar-Seg2  SnackBar      Retail   
4      2023-12-29  SN-013  SnBrand3  SnackBar-Seg2  SnackBar      Retail   
...           ...     ...       ...            ...       ...         ...   
191176 2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt      Retail   
191177 2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt      Retail   
191178 2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt    Discount   
191179 2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt    Discount   
191180 2023-12-29  YO-012  YoBrand1    Yogurt-Seg2    Yogurt    Discount   

            region  pack_type  price_unit  promotion_flag  ...  \
0         PL-North   

### 4.7. Null handling for 'is_npi_flag'

In [276]:
df['is_npi_flag'] = (df['lifecycle_stage'] == 'Introduction').astype(int)

In [277]:
 print (df['is_npi_flag'])

0         1
1         1
2         1
3         1
4         1
         ..
191176    1
191177    1
191178    1
191179    1
191180    1
Name: is_npi_flag, Length: 191181, dtype: int64


### 4.8. Null handling for 'lag_1' and 'lag_2'

In [278]:
df['lag_1'] = df['lag_1'].replace('', 0).fillna(0).astype(int)

In [279]:
df['lag_2'] = df['lag_2'].replace('', 0).fillna(0).astype(int)

### 4.9. Handling negative values for 'delivered_qty'

In [280]:
(df['delivered_qty'] < 0).sum()

np.int64(3)

In [281]:
def correct_delivered_qty (row):
    if row['delivered_qty'] <0 :
        return 0
    else: return row['delivered_qty']

df['delivered_qty'] = df.apply(correct_delivered_qty, axis=1)

In [282]:
print (df['delivered_qty'])

0         195.0
1         158.0
2         259.0
3         265.0
4         182.0
          ...  
191176    187.0
191177    165.0
191178    278.0
191179    137.0
191180    282.0
Name: delivered_qty, Length: 191181, dtype: float64


### 4.10. Handling negative values for 'price_unit'

In [283]:
(df['price_unit'] < 0).sum()

np.int64(0)

In [284]:
def correct_unit_price (row):
    if row['price_unit'] <0 :
        return 0
    else: return row['price_unit']

df['price_unit'] = df.apply(correct_unit_price, axis=1)

In [285]:
print (df['price_unit'])

0         3.65
1         6.75
2         3.49
3         7.46
4         2.25
          ... 
191176    4.02
191177    3.86
191178    4.83
191179    3.65
191180    1.97
Name: price_unit, Length: 191181, dtype: float64


### 4.11. Handling negative values for 'units_sold'

In [286]:
(df['units_sold'] < 0).sum()

np.int64(3)

In [287]:
def correct_unit_sold (row):
    if row['units_sold'] <0 :
        return 0
    else: return row['units_sold']

df['units_sold'] = df.apply(correct_unit_sold, axis=1)

In [288]:
print( df['units_sold'])

0         20.0
1         21.0
2         21.0
3         22.0
4         14.0
          ... 
191176    18.0
191177    15.0
191178    33.0
191179     9.0
191180    24.0
Name: units_sold, Length: 191181, dtype: float64


### 4.12. Null Handling values for 'price_avg'

In [289]:
df['price_unit'] = pd.to_numeric(df['price_unit'], errors='coerce').fillna(0)
df['units_sold'] = pd.to_numeric(df['units_sold'], errors='coerce').fillna(0)
df['revenue'] = df['price_unit'] * df['units_sold']
# This adds the column 'avg_price' directly to df
df['price_avg'] = (
    df.groupby('week_number')['revenue'].transform('sum') / 
    df.groupby('week_number')['units_sold'].transform('sum')
)

In [290]:
print (df['price_avg'])

0         5.249205
1         5.249205
2         5.249205
3         5.249205
4         5.249205
            ...   
191176    5.249205
191177    5.249205
191178    5.249205
191179    5.249205
191180    5.249205
Name: price_avg, Length: 191181, dtype: float64


### 4.13. Null Handling values for 'promo_intensity'

In [291]:
df['promo_intensity'] = df['price_avg'] - df['price_unit']
df['promo_intensity']

0         1.599205
1        -1.500795
2         1.759205
3        -2.210795
4         2.999205
            ...   
191176    1.229205
191177    1.389205
191178    0.419205
191179    1.599205
191180    3.279205
Name: promo_intensity, Length: 191181, dtype: float64

### 4.14. Recalculating stock_available

In [292]:
# 1. Ensure date is in datetime format to extract month/year easily
df['date'] = pd.to_datetime(df['date'])

# 2. Sort remains the same
df = df.sort_values(by=['sku', 'region','channel', 'brand','segment','category','pack_type','date'])

# 3. Reset the column
df['stock_available'] = 0

group_cols = ['sku', 'region','channel', 'brand','segment','category','pack_type']

for key, group in df.groupby(group_cols):
    running_stock = 0
    last_month = None
    
    for i in group.index:
        current_date = df.at[i, 'date']
        
        # LOGIC: If the month changes (which includes year changes), reset stock to 0
        if last_month is not None and current_date.month != last_month:
            running_stock = 0
            
        # Record the opening stock
        df.at[i, 'stock_available'] = running_stock
        
        # Calculate what's left for the next day
        running_stock += df.at[i, 'delivered_qty'] - df.at[i, 'units_sold']
        
        # Safety check: No negative stock
        if running_stock < 0:
            running_stock = 0
            
        # Update trackers for the next iteration in the loop
        last_month = current_date.month

In [293]:
print(df['stock_available'])

93900       0
93987     171
94361     265
94817     467
95282     613
         ... 
81601     673
82065     861
82294    1106
82519    1247
84127    1382
Name: stock_available, Length: 191181, dtype: int64


### 4.15. Recalculating stockout_flag

In [294]:
df['stockout_flag']= (df['stock_available']==0).astype(int)

In [295]:
print(df['stockout_flag'])

93900    1
93987    0
94361    0
94817    0
95282    0
        ..
81601    0
82065    0
82294    0
82519    0
84127    0
Name: stockout_flag, Length: 191181, dtype: int64


### 4.16. Recalculating 'stock_avg' and 'sales_total_monthly'

In [296]:
df['stock_avg'] = df.groupby(['sku', 'segment', 'pack_type', 'region', 'month_m','year_y'])['stock_available'].transform('mean')

In [297]:
df['stock_avg']

93900    413.55
93987    413.55
94361    413.55
94817    413.55
95282    413.55
          ...  
81601    559.30
82065    559.30
82294    559.30
82519    559.30
84127    559.30
Name: stock_avg, Length: 191181, dtype: float64

In [298]:
df['sales_total_monthly'] = df.groupby(['sku', 'segment', 'pack_type', 'region', 'month_m','year_y'])['units_sold'].transform('sum')

In [299]:
df['sales_total_monthly'] 

93900    333.0
93987    333.0
94361    333.0
94817    333.0
95282    333.0
         ...  
81601    414.0
82065    414.0
82294    414.0
82519    414.0
84127    414.0
Name: sales_total_monthly, Length: 191181, dtype: float64

### 4.17. Recalculating 'inventory_turnover_ratio'

In [300]:
import numpy as np
df['inventory_turnover_ratio'] = df.apply(
    lambda x: x['sales_total_monthly'] / x['stock_avg'] if x['stock_avg'] > 0 else 0, 
    axis=1
)
df['inventory_turnover_ratio'] = df['inventory_turnover_ratio'].replace([np.inf, -np.inf], 0)

print(df[['date', 'sku', 'stock_avg','region','inventory_turnover_ratio']].head())

            date     sku  stock_avg      region  inventory_turnover_ratio
93900 2022-07-11  JU-021     413.55  PL-Central                  0.805223
93987 2022-07-12  JU-021     413.55  PL-Central                  0.805223
94361 2022-07-16  JU-021     413.55  PL-Central                  0.805223
94817 2022-07-21  JU-021     413.55  PL-Central                  0.805223
95282 2022-07-26  JU-021     413.55  PL-Central                  0.805223


In [301]:
# Get a summary of the best-performing SKUs
best_movers = df.groupby('sku')['inventory_turnover_ratio'].mean().sort_values(ascending=False)

print("Top 5 Fast-Moving SKUs:")
print(best_movers.head(5))

print("\nTop 5 Slow-Moving SKUs (Potential Dead Stock):")
print(best_movers.tail(5))

Top 5 Fast-Moving SKUs:
sku
MI-006    1.002149
YO-020    0.939036
YO-029    0.929174
YO-024    0.918598
YO-016    0.912173
Name: inventory_turnover_ratio, dtype: float64

Top 5 Slow-Moving SKUs (Potential Dead Stock):
sku
SN-027    0.718017
JU-021    0.717065
MI-011    0.692586
MI-008    0.691027
MI-022    0.640722
Name: inventory_turnover_ratio, dtype: float64


In [302]:
peak_vs_normal = df.groupby('is_holiday_peak')['inventory_turnover_ratio'].mean()
print(peak_vs_normal)

is_holiday_peak
0    0.823897
1    0.817530
Name: inventory_turnover_ratio, dtype: float64


### 4.18. Droping the null columns

In [303]:
# List of columns that are mostly empty or redundant
cols_to_drop = [
    'predicted_units_sold', 'days_to_next_delivery', 'sku_age',
    'rolling_mean_4', 'rolling_std_4', 'momentum', 'target_next_week',
    'promo_rate', 'deliveries', 'avg_temp', 'inflation_index',
    'school_in_session', 'category_trend', 'event_score'
]
# Drop the columns
df.drop(columns=cols_to_drop, inplace=True)


In [304]:
df.to_csv(r'D:\Documents\Maloshree\DATA ANALYTICS CERTIFICATION-INCO\Capstone Project\Data\consolidated_data_load_filled.csv', index=False)